In [5]:
# Import Libraries and Load Dataset

import pandas as pd
import numpy as np

column_names = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes','land','wrong_fragment',
    'urgent','hot','num_failed_logins','logged_in','num_compromised','root_shell','su_attempted',
    'num_root','num_file_creations','num_shells','num_access_files','num_outbound_cmds','is_host_login',
    'is_guest_login','count','srv_count','serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate',
    'same_srv_rate','diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate','dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate','dst_host_serror_rate','dst_host_srv_serror_rate',
    'dst_host_rerror_rate','dst_host_srv_rerror_rate','class','difficulty'
]

train_df = pd.read_csv(r"C:\Users\yasha\Downloads\nsl-kdd\KDDTrain+.txt", header=None, names=column_names)
test_df  = pd.read_csv(r"C:\Users\yasha\Downloads\nsl-kdd\KDDTest+.txt", header=None, names=column_names)

print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)


Train Shape: (125973, 43)
Test Shape: (22544, 43)


In [6]:
# Remove leakage column & create binary target

train_df.drop("difficulty", axis=1, inplace=True)
test_df.drop("difficulty", axis=1, inplace=True)

train_df['attack_binary'] = (train_df['class'] != 'normal').astype(int)
test_df['attack_binary']  = (test_df['class'] != 'normal').astype(int)

# Split features and labels
X_train_raw = train_df.drop(['class','attack_binary'], axis=1)
y_train = train_df['attack_binary']

X_test_raw = test_df.drop(['class','attack_binary'], axis=1)
y_test = test_df['attack_binary']


In [13]:
from sklearn.preprocessing import LabelEncoder

categorical_cols = ['protocol_type', 'service', 'flag']
label_encoders = {}

X_train_enc = X_train_raw.copy()
X_test_enc  = X_test_raw.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_train_enc[col] = le.fit_transform(X_train_enc[col])
    X_test_enc[col]  = le.transform(X_test_enc[col])
    label_encoders[col] = le

print("Categorical Columns Encoded Successfully")
print(X_train_enc[categorical_cols].head())


Categorical Columns Encoded Successfully
   protocol_type  service  flag
0              1       20     9
1              2       44     9
2              1       49     5
3              1       24     9
4              1       24     9


In [14]:
from sklearn.preprocessing import StandardScaler

numerical_cols = [col for col in X_train_raw.columns if col not in categorical_cols]

scaler = StandardScaler()

X_train_scaled = X_train_raw.copy()
X_test_scaled = X_test_raw.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_enc[numerical_cols])
X_test_scaled[numerical_cols]  = scaler.transform(X_test_enc[numerical_cols])

print("Scaled Shapes:", X_train_scaled.shape, X_test_scaled.shape)


Scaled Shapes: (125973, 41) (22544, 41)


In [15]:
from sklearn.preprocessing import StandardScaler

numerical_cols = [col for col in X_train_enc.columns if col not in categorical_cols]

scaler = StandardScaler()

X_train_scaled = X_train_enc.copy()
X_test_scaled  = X_test_enc.copy()

X_train_scaled[numerical_cols] = scaler.fit_transform(X_train_enc[numerical_cols])
X_test_scaled[numerical_cols]  = scaler.transform(X_test_enc[numerical_cols])

print("Scaled Shapes:", X_train_scaled.shape, X_test_scaled.shape)


Scaled Shapes: (125973, 41) (22544, 41)


In [16]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", Counter(y_train))

sm = SMOTE(random_state=42)
X_train_bal, y_train_bal = sm.fit_resample(X_train_scaled, y_train)

print("After SMOTE :", Counter(y_train_bal))
print("Balanced Train Shape:", X_train_bal.shape)


Before SMOTE: Counter({0: 67343, 1: 58630})
After SMOTE : Counter({0: 67343, 1: 67343})
Balanced Train Shape: (134686, 41)


In [ ]:
# from sklearn.ensemble import RandomForestClassifier
# import matplotlib.pyplot as plt

# rf = RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1)
# rf.fit(X_train_bal, y_train_bal)

# importances = pd.Series(rf.feature_importances_, index=X_train_bal.columns)
# top_features = importances.sort_values(ascending=False).head(20)
# print("Top Selected Features:\n", top_features)

# # Reduced dataset
# X_train_reduced = X_train_bal[top_features.index]
# X_test_reduced = X_test_scaled[top_features.index]


In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1500, C=2.0, n_jobs=-1
    ),

    "Linear SVM": LinearSVC(
        C=1.0, max_iter=5000, random_state=42
    ),

    "Naive Bayes": GaussianNB(),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=None, min_samples_split=3, random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=350, max_depth=25,
        min_samples_leaf=2, random_state=42, n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=250, learning_rate=0.08,
        max_depth=3, random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=350, learning_rate=0.1, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
}

print("Models defined:", list(models.keys()))


Models defined: ['Logistic Regression', 'Linear SVM', 'Naive Bayes', 'Decision Tree', 'Random Forest', 'Gradient Boosting', 'XGBoost']


In [18]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

results_summary = []

print("\n MODEL EVALUATION STARTED \n")

# You can switch X_train_bal -> X_train_scaled, y_train_bal -> y_train
# if you want to see results WITHOUT SMOTE.
X_train_for_models = X_train_bal
y_train_for_models = y_train_bal

X_test_for_models  = X_test_scaled

for name, model in models.items():
    print(f"\n Training {name}...")

    model.fit(X_train_for_models, y_train_for_models)
    y_pred = model.predict(X_test_for_models)

    acc = accuracy_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred)
    rep = classification_report(y_test, y_pred, target_names=["Normal", "Attack"])

    results_summary.append([name, acc])

    print(f"\n📌 {name} Accuracy: {acc * 100:.2f}%")
    print("\nConfusion Matrix:")
    print(cm)
    print("\nClassification Report:")
    print(rep)
    print("\n")

print("\n MODEL EVALUATION COMPLETE \n")

summary_df = pd.DataFrame(results_summary, columns=["Model", "Accuracy"])
summary_df["Accuracy"] = summary_df["Accuracy"].apply(lambda x: f"{x*100:.2f}%")

print("\n OVERALL ACCURACY SUMMARY:")
display(summary_df.sort_values(by="Accuracy", ascending=False))



 MODEL EVALUATION STARTED 


 Training Logistic Regression...

📌 Logistic Regression Accuracy: 75.59%

Confusion Matrix:
[[9062  649]
 [4854 7979]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.93      0.77      9711
      Attack       0.92      0.62      0.74     12833

    accuracy                           0.76     22544
   macro avg       0.79      0.78      0.76     22544
weighted avg       0.81      0.76      0.75     22544




 Training Linear SVM...

📌 Linear SVM Accuracy: 75.47%

Confusion Matrix:
[[9052  659]
 [4870 7963]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.93      0.77      9711
      Attack       0.92      0.62      0.74     12833

    accuracy                           0.75     22544
   macro avg       0.79      0.78      0.75     22544
weighted avg       0.81      0.75      0.75     22544




 Training Naive Bayes...

📌 Naive Bayes Ac

,Model,Accuracy
6,XGBoost,80.35%
5,Gradient Boosting,79.31%
3,Decision Tree,78.43%
4,Random Forest,77.27%
2,Naive Bayes,77.16%
0,Logistic Regression,75.59%
1,Linear SVM,75.47%
